# Cryptography (CC4017) -- Week 11


## Chalenge 1

In the context of PKI:

### (a) 
Describe what is the accepted procedure of a client when receives a public-key certificate from a web server?

In this case, the server will send their identity, the public keys, the validy period of the CA, as well as some aditional meta-information all signed by a supposedly trusted CA. When receiving this information, the client should check if the identity is correct, that is, if the stated and given identity matches what is expected, in this case, the DNS name for the server matches what the client is waiting for. After that, the client checks if the validy period still holds, if the current time is between the given start and end time. The third step is to check if the meta information received makes sense in the context of the communication being done. After these 3 checks, the client should check if the CA is trustworthy to certify the public key the server sent. Lastly obtain the CA's public key, as well as verifying the signature in the certificate sent by the server.

### (b) 
Describe the process of certificate revocation and what are the possible reasons to apply it.Describe the process of certificate revocation and what are the possible reasons to apply it.

There are a myriad of valid reasons to revoke a certificate like the loss of secret key, a data breach or incorrect meta-data being sent. In case of this happening, the certificates need to be revoked, even if they still hold validity over their validity period. This is done using a CRL, certification revocation list, where a black-list of revoked certificates is published, containing all the certificates that are still valid but have been revoked, for browsers and protocols to check for before trusting a certificate. 

## Chalenge 2
In a Pailier’s scheme chanel, with private key n = 620496404349687915307910174617, we
intercepted the cyphered message
c = 358624662650643040547102063483144791182626860568435345308004
. Can you recover the original plaintext?

ĉ = (cΦ(n) mod n²)
^m   = (ĉ − 1)/n
m = ( ^m Φ(n)−1 mod n)

c = 358624662650643040547102063483144791182626860568435345308004

n = 620496404349687915307910174617 = 802829923639097 * 772886493240161 = p X q

n² = 385015787810891404063911270322113313569017982742671431096689

Φ(n) = (p - 1) * (q - 1) = 620496404349686339591493295360

ĉ = 358624662650643040547102063483144791182626860568435345308004^ 620496404349686339591493295360 mod 385015787810891404063911270322113313569017982742671431096689 = 385015787365373761792988167834554934384120851880533355749167

^m = 385015787365373761792988167834554934384120851880533355749166 / 620496404349687915307910174617 = 620496403631685942777789775198

406384760959582831173471620283 is the inverse of Φ(n)


 m = 620496403631685942777789775198 * 406384760959582831173471620283 mod 620496404349687915307910174617 = 455667







## Chalenge 3
Write the python/Sage procedures that behave as follows:

genPrivate(sz) that outputs a triple (n, p, q) in the conditions to be used as (n) the public
key of a Paillier’s scheme and n = pq, being p, q primes odd size sz bits;

voteYes(fileName,n) that append to ﬁle fileName a vote yes (=1) using Paillier’s scheme
and public key n.

voteNo(fileName,n) that append to ﬁle fileName a vote no (=0) using Paillier’s scheme
and public key n.

getResults(fileName,n,phi) that prints the result of the polling written in fileName
being n the public key used and phi the Euler’s totient value corresponding to the
public key.


In [34]:
from Crypto.Util import number
import random
import math

def extended_gcd(a, b):
    if b == 0:
        return a, 1, 0 
    else:
        gcd, x1, y1 = extended_gcd(b, a % b)
        x = y1
        y = x1 - (a // b) * y1
        return gcd, x, y


def genPrivate(sz):
    p = number.getPrime(sz)
    q = number.getPrime(sz)
    
    while p == q:
        q = number.getPrime(sz)
        
    n = p * q
    return n, p, q

def voteYes(fileName, n):
    m = 1
    n_square = n * n
    
    while True:
        r = random.randint(1, n-1)
        if (math.gcd(r, n) == 1):
            break
          
    c = (pow(1+n,m,n_square) *  pow(r,n,n_square)) % n_square

    with open(fileName, 'a') as file:
        file.write(f"{c}\n")

def voteNo(fileName, n):
    m = 0
    n_square = n * n
    
    while True:
        r = random.randint(1, n-1)
        if (math.gcd(r, n) == 1):
            break
          
    c = (pow(1+n,m,n_square) *  pow(r,n,n_square)) % n_square
    
    with open(fileName, 'a') as file:
        file.write(f"{c}\n")
        
def getResults(fileName,n,phi):
    n_squared = n * n 
    encrypted_sum = 1 
    num_lines = 0
    
    with open(fileName, 'r') as file:
        for line in file:
            line = line.strip()
            if not line.isdigit():  
                continue
            else:
                num_lines += 1
                c = int(line)  
                encrypted_sum = (encrypted_sum * c) % n_squared
    
    u = pow(encrypted_sum,phi,n_squared)
    mm = (u - 1) // (n)
    
    _, x ,_  = extended_gcd(phi,n) 
    inverse_phi =  x % n
    
    m = mm * inverse_phi % n 
    
    print(m,num_lines)
    return

n,p,q = genPrivate(11)
phi = (p-1)*(q-1)

voteYes("votes.txt",n)
voteYes("votes.txt",n)
voteNo("votes.txt",n)
voteNo("votes.txt",n)
voteYes("votes.txt",n)
voteYes("votes.txt",n)
voteNo("votes.txt",n)
voteNo("votes.txt",n)
getResults("votes.txt",n,phi)

  


4 8
